### General code

In [11]:
import gymnasium as gym
import numpy as np

from Simulation.suite_simple_trading.metrics import print_agent_performance
from stable_baselines3 import DQN, PPO, SAC
from Simulation.suite_simple_trading.simulation import run_evaluation, EvaluationResult

%load_ext autoreload
%autoreload 2

### Global Variables
RAW_DATA_PATH = '../../../data/2025_minute.csv'
DATA_SAVE_PATH = "../../../models/used_data"
DAYS_PER_EPISODE = 3
VAL_FRACTION = 0.1
TEST_FRACTION = 0.2  # Amount of data you reserve for testing.

BUFFER_DAYS = 3  # Amount of days that needs to be between different sets.

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
from Simulation.suite_simple_trading.pre_processing import clean_data
import pandas as pd
import os
from Simulation.suite_simple_trading.data_splitting import get_or_create_train_val_test_split
from Simulation.suite_simple_trading.model import ExtendedBatteryEnv

train_df, val_df_list, test_df_list = get_or_create_train_val_test_split(
    RAW_DATA_PATH,
    DATA_SAVE_PATH,
    DAYS_PER_EPISODE,
    VAL_FRACTION,
    TEST_FRACTION,
    BUFFER_DAYS,
)
test_df_combined = pd.concat(test_df_list, ignore_index=True)

--- Loading cached cleaned data: ../../../data/2025_minute_cleaned.pkl ---
--- Loading existing split from '../../../models/used_data' ---
Split loaded. Total Test Episodes: 10


## DQN

### Learn 100_000

In [20]:
### The model (Environment) used in all agents

train_env_dqn = ExtendedBatteryEnv(
    battery_capacity_mwh=10.0,
    charge_discharge_rate_mw=5.0,
    all_data=train_df,
    days_per_episode=DAYS_PER_EPISODE
)
dqn_agent = DQN(
    policy="MlpPolicy",
    env=train_env_dqn,
)
dqn_agent.learn(100_000)


In [21]:

test_df_combined = pd.concat(test_df_list, ignore_index=True)
test_env_dqn = ExtendedBatteryEnv(
    battery_capacity_mwh=10.0,
    charge_discharge_rate_mw=5.0,
    all_data=test_df_combined,
    days_per_episode=DAYS_PER_EPISODE
)
result_dqn: EvaluationResult = run_evaluation(test_env_dqn, dqn_agent, number_of_episodes=len(test_df), is_masked=False)

Starting episode 1/10
From 2025-04-30 00:00:00+00:00 to 2025-05-02 23:59:00+00:00
Finished with total (scaled) reward: 626.87
Starting episode 2/10
From 2025-09-23 00:00:00+00:00 to 2025-09-25 23:59:00+00:00
Finished with total (scaled) reward: 725.93
Starting episode 3/10
From 2025-03-01 00:00:00+00:00 to 2025-03-03 23:59:00+00:00
Finished with total (scaled) reward: 473.06
Starting episode 4/10
From 2025-01-05 00:00:00+00:00 to 2025-01-07 23:59:00+00:00
Finished with total (scaled) reward: 1105.56
Starting episode 5/10
From 2025-06-11 00:00:00+00:00 to 2025-06-13 23:59:00+00:00
Finished with total (scaled) reward: -524.95
Starting episode 6/10
From 2025-08-01 00:00:00+00:00 to 2025-08-03 23:59:00+00:00
Finished with total (scaled) reward: -593.68
Starting episode 7/10
From 2025-04-26 00:00:00+00:00 to 2025-04-28 23:59:00+00:00
Finished with total (scaled) reward: -567.18
Starting episode 8/10
From 2025-03-29 00:00:00+00:00 to 2025-03-31 23:59:00+00:00
Finished with total (scaled) rew

In [19]:
print_agent_performance(result_dqn.to_pandas(), "PPO Agent Compare")

  PERFORMANCE REPORT: PPO Agent Compare
  Pos. Profit Quarters : 885
  Neg. Profit Quarters : 934
----------------------------------------
  Success Rate (Active): 48.65%
  Mean Reward / active Quarter: €-33.9751
  Mean Daily Reward    : €-2060.0263
----------------------------------------
  Battery Cycles       : 105.05
  Profit per Cycle     : €-588.30
----------------------------------------
  'Trap' Quarters      : 55
  (Started good -> Ended bad)



### Learn 500_000

In [23]:
dqn_agent.learn(500_000)
result_dqn: EvaluationResult = run_evaluation(test_env_dqn, dqn_agent, number_of_episodes=len(test_df), is_masked=False)

KeyboardInterrupt: 

In [22]:
print_agent_performance(result_dqn.to_pandas(), "PPO Agent Compare")



  PERFORMANCE REPORT: PPO Agent Compare
  Pos. Profit Quarters : 468
  Neg. Profit Quarters : 212
----------------------------------------
  Success Rate (Active): 68.82%
  Mean Reward / active Quarter: €-2.0567
  Mean Daily Reward    : €-46.6176
----------------------------------------
  Battery Cycles       : 38.27
  Profit per Cycle     : €-36.55
----------------------------------------
  'Trap' Quarters      : 53
  (Started good -> Ended bad)



### Learn 1_000_000

In [25]:
train_env_dqn = ExtendedBatteryEnv(
    battery_capacity_mwh=10.0,
    charge_discharge_rate_mw=5.0,
    all_data=train_df,
    days_per_episode=DAYS_PER_EPISODE
)
dqn_agent = DQN(
    policy="MlpPolicy",
    env=train_env_dqn,
)
dqn_agent.learn(1_000_000)
test_env_dqn = ExtendedBatteryEnv(
    battery_capacity_mwh=10.0,
    charge_discharge_rate_mw=5.0,
    all_data=test_df_combined,
    days_per_episode=DAYS_PER_EPISODE
)

result_dqn: EvaluationResult = run_evaluation(test_env_dqn, dqn_agent, number_of_episodes=len(test_df), is_masked=False)

KeyboardInterrupt: 

## PPO

### Learn 100_000

In [26]:
### Learn 100_000

### Learn 500_000

In [29]:
train_env_ppo = ExtendedBatteryEnv(
    battery_capacity_mwh=10.0,
    charge_discharge_rate_mw=5.0,
    all_data=train_df,
    days_per_episode=DAYS_PER_EPISODE
)
ppo_agent = PPO(
    policy="MlpPolicy",
    env=train_env_ppo,
)
ppo_agent.learn(500_000)
test_env_ppo = ExtendedBatteryEnv(
    battery_capacity_mwh=10.0,
    charge_discharge_rate_mw=5.0,
    all_data=test_df_combined,
    days_per_episode=DAYS_PER_EPISODE
)
result_ppo: EvaluationResult = run_evaluation(test_env_ppo, ppo_agent, number_of_episodes=len(test_df), is_masked=False)


Starting episode 1/10
From 2025-04-30 00:00:00+00:00 to 2025-05-02 23:59:00+00:00
Finished with total (scaled) reward: 3341.80
Starting episode 2/10
From 2025-09-23 00:00:00+00:00 to 2025-09-25 23:59:00+00:00
Finished with total (scaled) reward: 1264.66
Starting episode 3/10
From 2025-03-01 00:00:00+00:00 to 2025-03-03 23:59:00+00:00
Finished with total (scaled) reward: 4606.93
Starting episode 4/10
From 2025-01-05 00:00:00+00:00 to 2025-01-07 23:59:00+00:00
Finished with total (scaled) reward: 8781.71
Starting episode 5/10
From 2025-06-11 00:00:00+00:00 to 2025-06-13 23:59:00+00:00
Finished with total (scaled) reward: 4908.51
Starting episode 6/10
From 2025-08-01 00:00:00+00:00 to 2025-08-03 23:59:00+00:00
Finished with total (scaled) reward: 7163.90
Starting episode 7/10
From 2025-04-26 00:00:00+00:00 to 2025-04-28 23:59:00+00:00
Finished with total (scaled) reward: 10393.35
Starting episode 8/10
From 2025-03-29 00:00:00+00:00 to 2025-03-31 23:59:00+00:00
Finished with total (scaled)

In [30]:
print_agent_performance(result_ppo.to_pandas(), "PPO Agent Compare")

  PERFORMANCE REPORT: PPO Agent Compare
  Pos. Profit Quarters : 810
  Neg. Profit Quarters : 111
----------------------------------------
  Success Rate (Active): 87.95%
  Mean Reward / active Quarter: €74.5818
  Mean Daily Reward    : €2289.6609
----------------------------------------
  Battery Cycles       : 53.82
  Profit per Cycle     : €1276.17
----------------------------------------
  'Trap' Quarters      : 65
  (Started good -> Ended bad)



## SAC

### 100_000 Learning steps

In [6]:
class ContinuousActionWrapper(gym.ActionWrapper):
    """
    Wraps a Discrete(3) environment to allow continuous actions for SAC.
    Maps range [-1, 1] to [0, 1, 2].
    """
    def __init__(self, env):
        super().__init__(env)
        # Change the action space to a continuous box
        self.action_space = gym.spaces.Box(low=-1.0, high=1.0, shape=(1,), dtype=np.float32)

    def action(self, continuous_action):
        """
        Maps the continuous output from SAC back to discrete integers.
        """
        if continuous_action < -0.33:
            return 2  # Discharge
        elif continuous_action > 0.33:
            return 1  # Charge
        else:
            return 0  # Idle

In [22]:
train_env = ExtendedBatteryEnv(
    battery_capacity_mwh=10.0,
    charge_discharge_rate_mw=5.0,
    all_data=train_df,
    days_per_episode=DAYS_PER_EPISODE
)
train_env_sac = ContinuousActionWrapper(train_env)

sac_agent = SAC(
    policy="MlpPolicy",
    env=train_env_sac,
)
sac_agent.learn(100_000)



In [23]:
test_env = ExtendedBatteryEnv(
    battery_capacity_mwh=10.0,
    charge_discharge_rate_mw=5.0,
    all_data=test_df_combined,
    days_per_episode=DAYS_PER_EPISODE
)
test_env_sac = ContinuousActionWrapper(test_env)
result_sac: EvaluationResult = run_evaluation(test_env_sac, sac_agent, number_of_episodes=len(test_df_list), is_masked=False)

Starting episode 1/10
From 2025-04-30 00:00:00+00:00 to 2025-05-02 23:59:00+00:00
Finished with total (scaled) reward: 4104.53
Starting episode 2/10
From 2025-09-23 00:00:00+00:00 to 2025-09-25 23:59:00+00:00
Finished with total (scaled) reward: 1863.88
Starting episode 3/10
From 2025-03-01 00:00:00+00:00 to 2025-03-03 23:59:00+00:00
Finished with total (scaled) reward: 4705.78
Starting episode 4/10
From 2025-01-05 00:00:00+00:00 to 2025-01-07 23:59:00+00:00
Finished with total (scaled) reward: 9823.41
Starting episode 5/10
From 2025-06-11 00:00:00+00:00 to 2025-06-13 23:59:00+00:00
Finished with total (scaled) reward: 5746.72
Starting episode 6/10
From 2025-08-01 00:00:00+00:00 to 2025-08-03 23:59:00+00:00
Finished with total (scaled) reward: 7204.51
Starting episode 7/10
From 2025-04-26 00:00:00+00:00 to 2025-04-28 23:59:00+00:00
Finished with total (scaled) reward: 10620.87
Starting episode 8/10
From 2025-03-29 00:00:00+00:00 to 2025-03-31 23:59:00+00:00
Finished with total (scaled)

In [18]:
print_agent_performance(result_sac.to_pandas(), "SAC Agent Compare")

  PERFORMANCE REPORT: SAC Agent Compare
  Pos. Profit Quarters : 756
  Neg. Profit Quarters : 113
----------------------------------------
  Success Rate (Active): 87.00%
  Mean Reward / active Quarter: €76.6089
  Mean Daily Reward    : €2219.1048
----------------------------------------
  Battery Cycles       : 49.63
  Profit per Cycle     : €1341.30
----------------------------------------
  'Trap' Quarters      : 71
  (Started good -> Ended bad)



In [21]:
print_agent_performance(result_sac.to_pandas(), "SAC Agent Compare")

  PERFORMANCE REPORT: SAC Agent Compare
  Pos. Profit Quarters : 1232
  Neg. Profit Quarters : 639
----------------------------------------
  Success Rate (Active): 65.85%
  Mean Reward / active Quarter: €38.6449
  Mean Daily Reward    : €2410.1530
----------------------------------------
  Battery Cycles       : 86.77
  Profit per Cycle     : €833.24
----------------------------------------
  'Trap' Quarters      : 68
  (Started good -> Ended bad)



In [24]:
print_agent_performance(result_sac.to_pandas(), "SAC Agent Compare")

  PERFORMANCE REPORT: SAC Agent Compare
  Pos. Profit Quarters : 1467
  Neg. Profit Quarters : 785
----------------------------------------
  Success Rate (Active): 65.14%
  Mean Reward / active Quarter: €31.9266
  Mean Daily Reward    : €2396.6237
----------------------------------------
  Battery Cycles       : 106.44
  Profit per Cycle     : €675.48
----------------------------------------
  'Trap' Quarters      : 71
  (Started good -> Ended bad)



## DDPG

## A2C